# 01 — Stockholm summer heat-event screening

This notebook runs the matching Python script, downloads **real SMHI hourly air temperature** for 2019–2023, and displays detected events. Requires network access from CSC. The first analysis uses a **user-selected screening definition** (3 consecutive days with daily maxima ≥25°C), **not** official SMHI heat warnings or corridor-scale heat stress.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Run Jupyter from the project root, or this cell locates the script relative to notebooks/.
root = Path.cwd()
if not (root / "scripts" / "01_stockholm_heatwaves.py").exists():
    root = root.parent
assert (root / "scripts" / "01_stockholm_heatwaves.py").exists(), "Open notebook from repo root or notebooks/"
sys.path.insert(0, str(root / "scripts"))
from importlib import import_module
heat = import_module("01_stockholm_heatwaves")
print("Project root:", root)

In [ ]:
YEARS = [2019, 2020, 2021, 2022, 2023]
THRESHOLD_C = 25.0
MIN_CONSECUTIVE_DAYS = 3
OUT = root / "data" / "processed" / "stockholm_heat"

hourly, daily, events = heat.run(
    years=YEARS, output_dir=OUT,
    threshold=THRESHOLD_C, min_days=MIN_CONSECUTIVE_DAYS
)
print(f"Hourly observations: {len(hourly):,}; events: {len(events)}")
display(events)

In [ ]:
display(daily.groupby("year").agg(observed_days=("eligible", "sum"), hot_days=("hot_day", "sum"), peak_temp_c=("tmax_c", "max")))
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily["local_date"], daily["tmax_c"], lw=0.8)
ax.axhline(THRESHOLD_C, linestyle="--", label="Screening threshold")
for _, ev in events.iterrows():
    ax.axvspan(pd.Timestamp(ev.start_date),pd.Timestamp(ev.end_date)+pd.Timedelta(days=1),alpha=0.15)
ax.set(xlabel="Date (Stockholm local)", ylabel="Daily max air temperature (°C)", title="Summer heat episodes — SMHI station")
ax.legend(); plt.show()

## Next steps
- Verify station selection and missing-data coverage in `data/processed/stockholm_heat/analysis_metadata.json`.
- Repeat with alternative thresholds and sensitivity tests.
- Align event hours with Marina’s available summer mobility periods and GTFS snapshots.
- Do **not** interpret point-station air temperature as street-level UTCI or individual exposure.